# ConvNeXt-Tiny buffer 0 확인 — 2단계 × 질감, 시드 0 × 5fold (G4 속도 시험 겸)

## 왜 돌리나
- 시간 블록 평가의 buffer 30프레임은 측정 근거 없이 정한 값이다. DINOv2 민감도 분석에서는 buffer를 0으로 줄여도 성능이 오르지 않았다 (AUROC 0.9956~0.9988로 거의 같음).
- 같은 결론이 본 모델(ConvNeXt)에서도 나오는지 1조합 × 1시드로 확인한다. 같으면 "buffer 30이면 이웃 프레임 효과가 결과에 끼어들지 않는다"는 근거가 된다.
- 동시에 G4 GPU 속도를 L4 기록(같은 조합·같은 시드, fold당 11.2~13.9분)과 비교한다.

## 10/13 실행(2단계 × 질감, buffer 30)과 비교
| | 10/13 (buffer 30) | 이번 (buffer 0) |
|---|---|---|
| 학습 사진 | 학습 구간 ∩ 2단계 통과 | **buffer로 빼 두었던 사진까지 학습** (fold당 45~112장 더) |
| 업데이트 횟수 | 2840 / 2460 / 2260 / 2580 / 2740 | **같음** (사진이 늘어난 효과와 더 오래 학습한 효과가 섞이지 않게) |
| 시험 사진, 엔진, 설정, 시드 | — | 같음 (시드 0) |

- 조합을 2단계 × 질감으로 고른 이유: 10/13 어댑터에서 분할 파일만 바꾸면 되고, buffer 30 결과(3시드)가 이미 있어 바로 비교할 수 있다.
- 한계: 1시드라 시드 간 흔들림(이 조합은 공통 사진에서 21~26장)보다 작은 차이는 판단할 수 없다.

## 1. 설정
Drive `산공종설/`에 아래 5개가 있어야 한다. **새로 올릴 건 `convnext_buffer0_patch_20261015.zip` 하나뿐**이다.

| 파일 | 내용 | 상태 |
|---|---|---|
| underfill_originals_4000.zip | 원본 4,000장 | 이미 있음 |
| resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip | 공통 분류 함수, folds.csv, 예산 기준 목록 | 이미 있음 |
| convnext_three_preproc_patch_20261005.zip | 질감 마스크 4,000장 | 이미 있음 |
| dinov2_grid_bundle_20261013.zip | 2단계 v3.2 목록, 베드 지도 마스크 4,000장 | 이미 있음 |
| convnext_buffer0_patch_20261015.zip | 이번 어댑터 + 엔진 복사본 + buffer 0 분할표 | **새로 올림** |

런타임은 **G4** (없으면 L4). 끊겨도 같은 RUN_NAME으로 위부터 다시 실행하면 이어서 한다 (라운드마다 저장).

In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
ORIGINAL_ZIP_NAME = 'underfill_originals_4000.zip'
BUNDLE_ZIP_NAME = 'resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip'
TEXTURE_ZIP_NAME = 'convnext_three_preproc_patch_20261005.zip'
GRID_ZIP_NAME = 'dinov2_grid_bundle_20261013.zip'
PATCH_ZIP_NAME = 'convnext_buffer0_patch_20261015.zip'
RUN_NAME = 'convnext_buffer0_20261015_run01'
IMG_SIZE = 512
BATCH = 16
ROUNDS = 20
SEEDS = [0]
FOLDS = [0, 1, 2, 3, 4]
RUN_TRAINING = True         # False면 입력 준비·점검만

In [ ]:
import torch, torchvision
from google.colab import drive
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name())
drive.mount('/content/drive')
OUTPUT = DRIVE_DIR / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)

## 2. 파일 풀기·무결성 점검
ZIP을 런타임 SSD(`/content`)에 푼다. Drive에서 바로 읽으면 느리다.
`--verify-only`는 학습 없이 5가지 분할(buffer 0)이 시험 구간과 겹치지 않는지, 업데이트 횟수가 옛 실험과 같은지만 확인한다.
- 확인할 것: `eligibility_audit.csv`의 `total_update_steps`가 fold0~4에서 2840/2460/2260/2580/2740 (옛 실험과 동일)

In [ ]:
import zipfile, hashlib, json, shutil, subprocess, sys, os
import pandas as pd
from IPython.display import display
LOCAL = Path('/content/convnext_buffer0_20261015')
LOCAL.mkdir(parents=True, exist_ok=True)
BUNDLE, ORIGINALS, TEXPATCH, GRID, PATCH = (LOCAL/n for n in ('bundle', 'originals', 'texpatch', 'grid', 'patch'))
CACHE = LOCAL / f'cache_{IMG_SIZE}'

def extract_zip(source, target):
    assert source.is_file(), f'파일 없음: {source}'
    local_zip = LOCAL / source.name
    if not local_zip.exists() or local_zip.stat().st_size != source.stat().st_size:
        shutil.copy2(source, local_zip)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for item in archive.infolist():
            assert (target/item.filename).resolve().is_relative_to(target.resolve()), 'ZIP 경로 오류'
        archive.extractall(target)
    print('준비:', source.name)

for name, target in ((BUNDLE_ZIP_NAME, BUNDLE), (TEXTURE_ZIP_NAME, TEXPATCH), (GRID_ZIP_NAME, GRID),
                     (PATCH_ZIP_NAME, PATCH), (ORIGINAL_ZIP_NAME, ORIGINALS)):
    extract_zip(DRIVE_DIR / name, target)
for folder in (BUNDLE, TEXPATCH, PATCH):
    hashes = json.loads((folder/'checksums.json').read_text())
    for name, expected in hashes.items():
        assert hashlib.sha256((folder/name).read_bytes()).hexdigest() == expected, name
    print('해시 일치:', folder.name, len(hashes), '개')
os.environ['PYTHONPATH'] = str(PATCH) + os.pathsep + str(BUNDLE) + os.pathsep + os.environ.get('PYTHONPATH', '')
BASE_COMMAND = [sys.executable, '-u', str(PATCH/'train_convnext_buffer0.py'),
    '--bundle', str(BUNDLE), '--texture', str(TEXPATCH/'texture'), '--grid', str(GRID),
    '--originals', str(ORIGINALS), '--cache', str(CACHE), '--output', str(OUTPUT),
    '--size', str(IMG_SIZE), '--batch', str(BATCH), '--rounds', str(ROUNDS)]
subprocess.run(BASE_COMMAND + ['--verify-only'], check=True)
audit = pd.read_csv(OUTPUT/'eligibility_audit.csv')
display(audit)
assert audit.groupby('fold').total_update_steps.first().tolist() == [2840, 2460, 2260, 2580, 2740]
display(pd.read_csv(OUTPUT/'coverage.csv'))

## 3. 입력 캐시 만들기 + 눈으로 확인
4,000장 × 질감 마스크 1벌을 512×512로 줄여 SSD에 저장한다 (약 3GB, 10분 안팎. CPU 작업이라 GPU 종류와 무관).
- 왜 미리 만드나: 20라운드 동안 같은 사진을 수십 번 읽는데, 매번 JPG를 풀고 마스크를 칠하면 GPU가 놀게 된다.
- 마스크 밖은 회색 RGB(124,116,104)로 칠한다. 기존 실험과 같은 색이다.
- 아래 그림에서 확인할 것: 베드가 보이는 사진(아래 두 줄)에서 베드 부분이 회색으로 가려졌는지.

In [ ]:
# 캐시 표시 파일(complete.json)이 이미 있으면 다시 만들지 않는다 (아래 6번 설명 참고)
if not (CACHE/'complete.json').exists():
    subprocess.run(BASE_COMMAND + ['--prepare-only'], check=True)
import numpy as np
import matplotlib.pyplot as plt
inputs = pd.read_csv(OUTPUT/'input_manifest.csv')
arrays = {n: np.load(CACHE/f'{n}.npy', mmap_mode='r') for n in ('texture',)}
picks = []
for clean in (1, 0):
    for label in ('normal', 'underfill_50'):
        g = inputs[(inputs.common == 1) & (inputs.clean_old == clean) & (inputs.label == label)]
        picks.append((g.index[len(g)//2], 'clean' if clean else 'bed visible'))
fig, axes = plt.subplots(len(picks), 1, figsize=(4, 4*len(picks)), squeeze=False)
for i, (idx, kind) in enumerate(picks):
    r = inputs.iloc[idx]
    for j, n in enumerate(('texture',)):
        axes[i, j].imshow(arrays[n][idx].transpose(1, 2, 0)); axes[i, j].axis('off')
        axes[i, j].set_title(f'{n}: {r.label} #{r.frame_index} ({kind})')
fig.tight_layout(); plt.show(); plt.close(fig)

## 4. 실행·결과 표시 함수
긴 로그는 Drive의 `train_seed*.log`에 저장하고 화면에는 fold 완료 줄만 띄운다.
결과 표에서 볼 것 (순위 비교는 `common` 3,039장에서만):
- `common`: 세 선별 모두 통과한 공통 사진. DINOv2 격자 A와 같은 사진이다.
- `common_clean` / `common_bed_visible`: 공통 사진을 깨끗한 사진 2,648 / 베드가 보이는 사진 391로 나눔. **마스크 차이는 베드가 보이는 쪽에서 나야 한다** [추정: 격자 결과 기반].
- `own_stage2`: 2단계가 판정한 3,062장 (운영 성능).
- `none_pass`: 세 선별 모두 보류한 539장. 정답률이 낮아도 정상이다 (판정 불가 사진이라 선별이 필요한 근거).

In [ ]:
def export_results(suffix):
    destination = DRIVE_DIR / f'{RUN_NAME}_{suffix}_results.zip'
    temp = LOCAL / destination.name
    with zipfile.ZipFile(temp, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix.lower() in {'.csv', '.json', '.log', '.png', '.md'}:
                archive.write(path, path.relative_to(OUTPUT).as_posix())
    shutil.copy2(temp, destination)
    print('공유할 결과 ZIP:', destination)
    return destination

def show_results(seed):
    progress = json.loads((OUTPUT/'progress.json').read_text())
    assert seed in progress['fully_completed_seeds'], '이 시드의 5회가 아직 완료되지 않았습니다.'
    print(f'시드 {seed} 완료 — 전체 {progress["completed_runs"]}/5회')
    s = pd.read_csv(OUTPUT/'summary.csv')
    display(s[s.seed == seed][['version', 'group', 'n', 'accuracy', 'underfill_recall', 'normal_recall', 'auroc', 'fp', 'fn']])
    m = pd.read_csv(OUTPUT/'across_seeds.csv')
    display(m[(m.group == 'common') & m.metric.isin(['accuracy', 'fp', 'fn', 'auroc'])])
    return export_results(f'seed{seed}')

def train_seed(seed):
    progress = OUTPUT/'progress.json'
    if progress.exists() and seed in json.loads(progress.read_text())['fully_completed_seeds']:
        print(f'시드 {seed}는 이미 5회 완료 — 학습 없이 결과만 다시 표시합니다.')
        return show_results(seed)
    log_path = OUTPUT / f'train_seed{seed}.log'
    command = BASE_COMMAND + ['--seeds', str(seed), '--folds', *map(str, FOLDS)]
    print(f'시드 {seed}: 1조건 × 5fold = 5회 시작. 로그: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as logfile:
        logfile.write('\n실행 명령: '+json.dumps(command, ensure_ascii=False)+'\n')
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            logfile.write(line); logfile.flush()
            if f'round {ROUNDS}/{ROUNDS}:' in line or '이미 완료' in line:
                print(line.rstrip(), flush=True)
        return_code = process.wait()
    if return_code:
        print(log_path.read_text(encoding='utf-8')[-7000:])
        raise RuntimeError(f'학습 오류. 기존 저장은 유지됩니다. 로그: {log_path}')
    return show_results(seed)

## 5. 시드 0 실행 (5회)
끝나면 표와 `..._seed0_results.zip`이 나온다. 바로 아래 6번 셀이 G4 속도를 L4 기록과 비교한다.

In [ ]:
assert SEEDS == [0] and FOLDS == [0, 1, 2, 3, 4]
if RUN_TRAINING:
    SEED0_ZIP = train_seed(0)
else:
    print('입력 준비만 완료했습니다. 학습하지 않았습니다.')

## 6. G4 속도: L4 기록과 비교
L4 기록 = 10/13 실행의 같은 조합·같은 시드 fold별 학습 시간. 업데이트 횟수가 같으므로 시간 차이는 GPU 차이다 (캐시 준비 시간은 제외).

In [ ]:
L4_MIN = {0: 13.9, 1: 12.1, 2: 11.2, 3: 12.6, 4: 13.2}   # 10/13 v4_s2_texture seed0 complete.json
rows = []
for k in FOLDS:
    c = json.loads((OUTPUT/'v4b0_s2_texture'/'seed0'/f'fold{k}'/'complete.json').read_text())
    rows.append(dict(fold=k, 이번_분=round(c['seconds']/60, 1), L4_분=L4_MIN[k], 배속=round(L4_MIN[k]/(c['seconds']/60), 2)))
speed = pd.DataFrame(rows); display(speed)
print('GPU:', json.loads((OUTPUT/'environment.json').read_text())['gpu'], '| 평균 배속', round(speed.배속.mean(), 2))
speed.to_csv(OUTPUT/'speed_vs_L4.csv', index=False, encoding='utf-8-sig')
FINAL_ZIP = export_results('seed0_with_speed')

## 7. 공유
`..._seed0_with_speed_results.zip`을 받아 오면 된다. buffer 30 결과(10/13, 3시드)와의 같은 사진 비교는 로컬에서 계산한다.

## 8. 런타임 반납
결과 ZIP이 Drive에 있는지 확인한 뒤 실행. 학습 중에는 실행하지 말 것.

In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()